# Radioactive-Decay-Inspired RNG: NZ Lotto Powerball Dataset Analysis

**Abstract.** We characterise 1,869 NZ Lotto Powerball draws (2001-02-17 → 2026-04-22) using statistical tests drawn directly from the physics of radioactive decay. The inter-arrival gaps between successive appearances of each number are tested against an Exponential distribution; draw-count-in-window is tested against Poisson; the full sequence is tested for independence. We then introduce a *decay-mimicking generator*: 40 virtual radioactive sources (one per lottery number) each with an empirical decay rate λₙ; the 6 sources whose first emission arrives earliest form the ticket. We validate this generator against uniform, the existing Gaussian-weighted picker, and the historical empirical distribution. **No predictive claim is made.** A fair lottery is i.i.d. by design; the analysis asks whether the data looks like Poisson emissions, and demonstrates a physics-grounded RNG pattern with pedagogical value only.

## §1 — Setup & Data Load

In [ ]:
import json
import secrets
import itertools
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from scipy import stats

rng = np.random.default_rng(seed=42)

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120

DATA_PATH = Path('../../frontend/public/results.json')

In [ ]:
with open(DATA_PATH) as f:
    raw = json.load(f)

draws = pd.DataFrame(raw)
draws['date'] = pd.to_datetime(draws['date'])
draws = draws.sort_values('date').reset_index(drop=True)

# Sanity checks
assert all(draws['numbers'].apply(len) == 6), 'Not all draws have 6 numbers'
assert all(draws['numbers'].apply(lambda x: len(set(x))) == 6), 'Duplicate numbers in a draw'
assert draws['numbers'].apply(lambda x: all(1 <= n <= 40 for n in x)).all(), 'Number out of [1,40]'
assert draws['powerball'].between(1, 10).all(), 'Powerball out of [1,10]'

N_DRAWS = len(draws)
print(f'Total draws : {N_DRAWS}')
print(f'Date range  : {draws.date.min().date()} → {draws.date.max().date()}')

# Long form: one row per (draw_index, number)
long = draws.explode('numbers').rename(columns={'numbers': 'number'})
long['number'] = long['number'].astype(int)
print(f'Long-form rows (should be {N_DRAWS}×6 = {N_DRAWS*6}): {len(long)}')

## §2 — Dataset Characterisation

In [ ]:
# Per-number frequency tables
freq_main = long.groupby('number').size().rename('count').reset_index()
freq_main['expected'] = N_DRAWS * 6 / 40
freq_main['freq_rate'] = freq_main['count'] / N_DRAWS  # appearances per draw

freq_pb = draws.groupby('powerball').size().rename('count').reset_index()
freq_pb.columns = ['number', 'count']
freq_pb['expected'] = N_DRAWS / 10

print('--- Main numbers (1–40) ---')
print(freq_main.describe())
print('\n--- Powerball (1–10) ---')
print(freq_pb)

In [ ]:
# Chi-square goodness-of-fit vs uniform
chi2_main, p_main = stats.chisquare(freq_main['count'])
chi2_pb, p_pb = stats.chisquare(freq_pb['count'])

print(f'Main numbers  χ²={chi2_main:.3f}  dof={len(freq_main)-1}  p={p_main:.4f}')
print(f'Powerball     χ²={chi2_pb:.3f}   dof={len(freq_pb)-1}    p={p_pb:.4f}')

# Bonferroni-corrected individual tests (40 tests → threshold α/40)
alpha_bonf = 0.05 / 40
individual = []
for n in range(1, 41):
    cnt = freq_main.loc[freq_main.number == n, 'count'].values[0]
    expected = N_DRAWS * 6 / 40
    chi2_i = (cnt - expected)**2 / expected
    # Approximate with 1 dof (Poisson deviance, not full chi2, but good for flagging)
    p_i = stats.chi2.sf(chi2_i, df=1)
    if p_i < alpha_bonf:
        individual.append((n, cnt, round(p_i, 5)))

if individual:
    print(f'\nBonferroni significant (α={alpha_bonf:.4f}):', individual)
else:
    print(f'\nNo number is individually significant at Bonferroni-corrected α={alpha_bonf:.4f}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Main numbers
ax = axes[0]
ax.bar(freq_main['number'], freq_main['count'], color='steelblue', alpha=0.75, label='Observed')
ax.axhline(freq_main['expected'].iloc[0], color='tomato', lw=1.5, ls='--', label='Expected (uniform)')
ax.set_xlabel('Ball number (1–40)')
ax.set_ylabel('Appearances')
ax.set_title(f'Main number frequency  (χ²={chi2_main:.1f}, p={p_main:.3f})')
ax.legend()

# Powerball
ax = axes[1]
ax.bar(freq_pb['number'], freq_pb['count'], color='darkorange', alpha=0.75, label='Observed')
ax.axhline(freq_pb['expected'].iloc[0], color='tomato', lw=1.5, ls='--', label='Expected (uniform)')
ax.set_xlabel('Powerball (1–10)')
ax.set_ylabel('Appearances')
ax.set_title(f'Powerball frequency  (χ²={chi2_pb:.1f}, p={p_pb:.3f})')
ax.legend()

plt.tight_layout()
plt.savefig('freq_distribution.png', bbox_inches='tight')
plt.show()

### §2.1 — Inter-Arrival Gaps (The Decay Analogue)

In radioactive decay, the time between successive emissions from a single source follows an **Exponential distribution** (continuous) or **Geometric distribution** (discrete). If NZ Lotto numbers are drawn i.i.d. uniformly, the gap *g* between successive appearances of number *n* is Geometric with parameter *p* = 6/40 = 0.15. We test this.

In [ ]:
# Compute gap sequences per number
draw_indices = {}  # number -> sorted list of draw indices it appeared in
for n in range(1, 41):
    mask = long['number'] == n
    idxs = long[mask].index.tolist()  # index in long-form; we want draw-level index

# Use draw-level ordering
draws['draw_idx'] = range(N_DRAWS)
long2 = draws.explode('numbers').rename(columns={'numbers': 'number'})
long2['number'] = long2['number'].astype(int)

gaps_all = []  # pooled across all numbers
gaps_per_num = {}

for n in range(1, 41):
    draw_idxs = long2.loc[long2['number'] == n, 'draw_idx'].sort_values().values
    gaps = np.diff(draw_idxs)  # gap = number of draws between appearances
    gaps_per_num[n] = gaps
    gaps_all.extend(gaps)

gaps_all = np.array(gaps_all)
print(f'Total gap observations: {len(gaps_all)}')
print(f'Gap stats: mean={gaps_all.mean():.2f}, median={np.median(gaps_all):.0f}, std={gaps_all.std():.2f}')
print(f'Theoretical geometric mean (p=6/40): {1/(6/40):.2f}')

In [ ]:
# Fit Geometric distribution (discrete, p estimated from data)
p_hat = 1 / gaps_all.mean()  # MLE for geometric is 1/mean
p_theoretical = 6 / 40

print(f'Geometric p (empirical MLE): {p_hat:.4f}')
print(f'Geometric p (theoretical, uniform lottery): {p_theoretical:.4f}')

# KS test against fitted geometric
# Use exponential as continuous approximation
ks_stat, ks_p = stats.kstest(gaps_all, 'expon', args=(0, gaps_all.mean()))
print(f'\nKS test vs Exponential(λ̂): stat={ks_stat:.4f}, p={ks_p:.4f}')

# QQ plot
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Histogram + fitted exponential
ax = axes[0]
ax.hist(gaps_all, bins=60, density=True, alpha=0.6, color='steelblue', label='Empirical gaps')
x = np.linspace(0, gaps_all.max(), 500)
scale = gaps_all.mean()
ax.plot(x, stats.expon.pdf(x, scale=scale), 'r-', lw=2, label=f'Exp(λ̂={1/scale:.3f})')
ax.plot(x, stats.expon.pdf(x, scale=1/p_theoretical), 'g--', lw=1.5, label=f'Exp(p_theory={p_theoretical})')
ax.set_xlabel('Gap (draws between appearances)')
ax.set_ylabel('Density')
ax.set_title('Inter-arrival gap distribution')
ax.legend()

# QQ plot
ax = axes[1]
(osm, osr), (slope, intercept, r) = stats.probplot(gaps_all, dist='expon', sparams=(0, scale))
ax.plot(osm, osr, 'o', alpha=0.3, markersize=2, color='steelblue')
ax.plot(osm, slope * np.array(osm) + intercept, 'r-', lw=1.5)
ax.set_xlabel('Theoretical quantiles (Exponential)')
ax.set_ylabel('Sample quantiles')
ax.set_title(f'QQ plot — gaps vs Exponential  (R²={r**2:.4f})')

plt.tight_layout()
plt.savefig('gap_distribution.png', bbox_inches='tight')
plt.show()

In [ ]:
# Per-number empirical lambda (appearances per draw) — the 'decay rate'
lambdas_main = np.array([
    freq_main.loc[freq_main.number == n, 'freq_rate'].values[0]
    for n in range(1, 41)
])

# Laplace smoothing so no number has λ=0
lambdas_main_smooth = (freq_main['count'].values + 1) / (N_DRAWS * 6 + 40)
# Normalize to sum = 6/40 per position (decay rates proportional to frequency)
# Actually keep raw for simulation; they'll be used as Exp rate params.

print('λ range (main, smoothed):', lambdas_main_smooth.min(), '–', lambdas_main_smooth.max())
print('λ mean:', lambdas_main_smooth.mean())

# Powerball lambdas
lambdas_pb_smooth = (freq_pb['count'].values + 1) / (N_DRAWS + 10)

### §2.2 — Count-in-Window (Poisson Check)

For a radioactive source with rate λ, the count of emissions in a fixed time window *W* follows **Poisson(λW)**. For each number *n* with appearance rate λₙ, the count in a window of *W*=50 draws should follow Poisson(λₙ · 50) ≈ Poisson(7.5) under uniformity.

In [ ]:
W = 50

# Compute sliding window counts for all numbers
number_matrix = np.zeros((N_DRAWS, 40), dtype=int)  # row=draw, col=number-1
for i, row in draws.iterrows():
    for n in row['numbers']:
        number_matrix[i, n - 1] = 1

window_counts = []
for start in range(0, N_DRAWS - W + 1, W):  # non-overlapping
    window = number_matrix[start:start + W]
    window_counts.extend(window.sum(axis=0).tolist())

window_counts = np.array(window_counts)
lambda_expected = (6 / 40) * W  # theoretical under uniformity

print(f'Window size W={W}, non-overlapping windows: {(N_DRAWS // W)}')
print(f'Total window-number observations: {len(window_counts)}')
print(f'Theoretical Poisson λ = (6/40)×{W} = {lambda_expected:.2f}')
print(f'Empirical mean: {window_counts.mean():.2f}, var: {window_counts.var():.2f}')
print(f'(Poisson has mean ≈ variance; ratio: {window_counts.mean()/window_counts.var():.3f})')

In [ ]:
# Chi-square test of count-in-window vs Poisson(lambda_expected)
obs_vals, obs_counts = np.unique(window_counts, return_counts=True)
k_min, k_max = obs_vals.min(), obs_vals.max()
all_k = np.arange(k_min, k_max + 1)
expected_probs = stats.poisson.pmf(all_k, mu=lambda_expected)
expected_counts = expected_probs * len(window_counts)

# Align observed to all_k
obs_aligned = np.array([obs_counts[obs_vals == k][0] if k in obs_vals else 0 for k in all_k])

# Merge small expected bins (< 5) at tails
chi2_wnd, p_wnd = stats.chisquare(obs_aligned, f_exp=expected_counts)
print(f'Count-in-window χ² vs Poisson({lambda_expected:.2f}): χ²={chi2_wnd:.3f}, p={p_wnd:.4f}')

fig, ax = plt.subplots(figsize=(9, 4))
ax.bar(all_k, obs_aligned / len(window_counts), alpha=0.6, color='steelblue', label='Observed')
ax.plot(all_k, expected_probs, 'ro-', markersize=5, label=f'Poisson({lambda_expected:.2f})')
ax.set_xlabel(f'Count of appearances in {W}-draw window')
ax.set_ylabel('Proportion')
ax.set_title(f'Count-in-window distribution  (χ²={chi2_wnd:.1f}, p={p_wnd:.3f})')
ax.legend()
plt.tight_layout()
plt.savefig('poisson_window.png', bbox_inches='tight')
plt.show()

### §2.3 — Independence Tests

In [ ]:
# Lag autocorrelation for indicator series of select numbers
MAX_LAG = 20
focus_numbers = [7, 13, 23, 38]  # representative spread

fig, axes = plt.subplots(1, len(focus_numbers), figsize=(14, 3), sharey=True)
for ax, n in zip(axes, focus_numbers):
    indicator = number_matrix[:, n - 1].astype(float)
    acf = [np.corrcoef(indicator[:-lag], indicator[lag:])[0, 1] for lag in range(1, MAX_LAG + 1)]
    se = 1.96 / np.sqrt(N_DRAWS)
    ax.bar(range(1, MAX_LAG + 1), acf, color='steelblue', alpha=0.7)
    ax.axhline(se, color='tomato', ls='--', lw=1)
    ax.axhline(-se, color='tomato', ls='--', lw=1)
    ax.axhline(0, color='black', lw=0.5)
    ax.set_title(f'ACF — number {n}')
    ax.set_xlabel('Lag (draws)')
    if ax == axes[0]:
        ax.set_ylabel('Autocorrelation')

plt.suptitle('Lag autocorrelation of indicator series (dashed = 95% CI under independence)', y=1.02)
plt.tight_layout()
plt.savefig('autocorrelation.png', bbox_inches='tight')
plt.show()

In [ ]:
# Wald-Wolfowitz runs test on a pooled hot/cold indicator
# Use number 23 (near median frequency) as a representative stream
def runs_test(x):
    """Wald-Wolfowitz runs test. Returns (z_stat, p_value)."""
    median = np.median(x)
    above = (x > median).astype(int)
    n1 = above.sum()
    n2 = len(above) - n1
    runs = 1 + np.sum(np.diff(above) != 0)
    expected_runs = (2 * n1 * n2) / (n1 + n2) + 1
    var_runs = (2 * n1 * n2 * (2 * n1 * n2 - n1 - n2)) / ((n1 + n2)**2 * (n1 + n2 - 1))
    z = (runs - expected_runs) / np.sqrt(var_runs)
    p = 2 * stats.norm.sf(abs(z))
    return runs, expected_runs, z, p

print('Runs test results for indicator streams:')
for n in focus_numbers:
    indicator = number_matrix[:, n - 1].astype(float)
    runs, exp_r, z, p = runs_test(indicator)
    print(f'  Number {n:2d}: runs={runs}, expected={exp_r:.1f}, z={z:.3f}, p={p:.4f}')

## §3 — Decay-Mimicking Generator

**Physical model.** A sealed radiation detector contains 40 sources (one per ball). Each source *n* emits particles at rate λₙ (its historical appearance rate per draw, Laplace-smoothed). Inter-emission times follow **Exponential(λₙ)**.

**Selection rule.** For each ticket, sample one waiting time τₙ ~ Exp(λₙ) per source. The 6 sources whose particles arrive first (smallest τ) are drawn.

**Mathematical equivalence note.** This is provably equivalent to *weighted sampling without replacement* with weights proportional to λₙ. The decay framing is a physical metaphor that maps well onto the Poisson process; it does not add statistical power beyond the weighting it encodes.

$$\tau_n \sim \text{Exp}(\lambda_n), \quad \text{ticket} = \operatorname*{argsort}(\tau)[:6] + 1$$

In [ ]:
def decay_draw(lam_main, lam_pb, rng_):
    """Sample one ticket from the decay-mimicking generator."""
    tau_main = rng_.exponential(scale=1.0 / lam_main)  # shape (40,)
    selected = np.argsort(tau_main)[:6] + 1             # 1-indexed numbers
    tau_pb   = rng_.exponential(scale=1.0 / lam_pb)    # shape (10,)
    pb       = np.argmin(tau_pb) + 1                    # single powerball 1-indexed
    return sorted(selected.tolist()), int(pb)

# Test single draw
numbers_test, pb_test = decay_draw(lambdas_main_smooth, lambdas_pb_smooth, rng)
print(f'Sample decay ticket: {numbers_test} | Powerball: {pb_test}')

## §4 — Validation

We generate 10,000 tickets from four generators and compare their distributions.

In [ ]:
N_SIM = 10_000

# --- Generator A: Decay (ours) ---
decay_tickets = [decay_draw(lambdas_main_smooth, lambdas_pb_smooth, rng) for _ in range(N_SIM)]
decay_nums = [n for t, _ in decay_tickets for n in t]

# --- Generator B: Uniform random (baseline) ---
uniform_nums = []
for _ in range(N_SIM):
    nums = rng.choice(np.arange(1, 41), size=6, replace=False)
    uniform_nums.extend(nums.tolist())

# --- Generator C: Gaussian-weighted (port of frontend/src/utils.ts) ---
def box_muller(rng_):
    u, v = 0.0, 0.0
    while u == 0: u = rng_.uniform()
    while v == 0: v = rng_.uniform()
    return np.sqrt(-2 * np.log(u)) * np.cos(2 * np.pi * v)

def gaussian_weighted_choice(freq_map, mean, stddev, rng_):
    target = box_muller(rng_) * stddev + mean
    weights = []
    for i in range(1, 41):
        base = freq_map.get(i, 0)
        proximity = np.exp(-((i - target) ** 2) / (2 * 4 ** 2))
        weights.append((base + 1) * (1 + proximity * 5))
    weights = np.array(weights)
    weights /= weights.sum()
    return rng_.choice(np.arange(1, 41), p=weights)

freq_map = {int(r.number): int(r['count']) for _, r in freq_main.iterrows()}

def gaussian_ticket(freq_map, rng_, leaning='mixed'):
    mean_map = {'left': 10, 'middle': 20, 'right': 30, 'mixed': 20}
    stddev_map = {'left': 8, 'middle': 8, 'right': 8, 'mixed': 15}
    mean = mean_map[leaning]
    stddev = stddev_map[leaning]
    nums = []
    while len(nums) < 6:
        n = gaussian_weighted_choice(freq_map, mean, stddev, rng_)
        if n not in nums:
            nums.append(n)
    # Powerball: Gaussian(5.5, 2.2), clamped to [1,10]
    pb_val = round(box_muller(rng_) * 2.2 + 5.5)
    pb = max(1, min(10, int(pb_val)))
    return sorted(nums), pb

gaussian_tickets = [gaussian_ticket(freq_map, rng) for _ in range(N_SIM)]
gaussian_nums = [n for t, _ in gaussian_tickets for n in t]

# --- Historical empirical distribution ---
historical_nums = long2['number'].tolist()

print('Generated ticket pools:')
print(f'  Decay    : {len(decay_nums)} numbers')
print(f'  Uniform  : {len(uniform_nums)} numbers')
print(f'  Gaussian : {len(gaussian_nums)} numbers')
print(f'  Historical: {len(historical_nums)} numbers')

In [ ]:
# Per-number frequency for each generator
def count_freq(nums_list, total):
    arr = np.zeros(40)
    for n in nums_list:
        arr[int(n) - 1] += 1
    return arr / total  # proportion per draw

f_decay    = count_freq(decay_nums, N_SIM)
f_uniform  = count_freq(uniform_nums, N_SIM)
f_gaussian = count_freq(gaussian_nums, N_SIM)
f_hist     = count_freq(historical_nums, N_DRAWS)

# Chi-square vs uniform for each generated distribution
expected_counts_gen = np.full(40, N_SIM * 6 / 40)

results = {}
for label, nums in [('Decay', decay_nums), ('Uniform', uniform_nums), ('Gaussian', gaussian_nums)]:
    obs = np.array([nums.count(n) for n in range(1, 41)], dtype=float)
    chi2, p = stats.chisquare(obs)
    results[label] = {'chi2': chi2, 'p': p}

# KL divergence (nats) — generator vs historical, generator vs uniform
f_uniform_theoretical = np.full(40, 6 / 40)

def kl_div(p, q):
    p = p / p.sum()
    q = q / q.sum()
    return np.sum(p * np.log(p / (q + 1e-12) + 1e-12))

for label, f_gen in [('Decay', f_decay), ('Uniform', f_uniform), ('Gaussian', f_gaussian)]:
    kl_hist = kl_div(f_gen, f_hist)
    kl_unif = kl_div(f_gen, f_uniform_theoretical)
    results[label]['kl_vs_hist'] = kl_hist
    results[label]['kl_vs_unif'] = kl_unif

print('\n=== Validation Summary ===')
print(f'{"Generator":<12} {"χ²":>8} {"p":>8} {"KL_hist":>10} {"KL_unif":>10}')
print('-' * 52)
for label, r in results.items():
    print(f'{label:<12} {r["chi2"]:>8.2f} {r["p"]:>8.4f} {r["kl_vs_hist"]:>10.6f} {r["kl_vs_unif"]:>10.6f}')

In [ ]:
# Side-by-side frequency comparison plot
fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)
x = np.arange(1, 41)
width = 0.35

pairs = [
    ('Decay generator', f_decay, 'steelblue'),
    ('Uniform random', f_uniform, 'seagreen'),
    ('Gaussian-weighted', f_gaussian, 'darkorange'),
]

for ax, (label, f_gen, color) in zip(axes, pairs):
    ax.bar(x - width/2, f_gen, width, label=label, color=color, alpha=0.7)
    ax.bar(x + width/2, f_hist, width, label='Historical empirical', color='slategray', alpha=0.55)
    ax.axhline(6/40, color='tomato', ls='--', lw=1, label='Uniform expected')
    ax.set_ylabel('Proportion per draw')
    ax.legend(loc='upper right', fontsize=8)
    ax.set_title(label)

axes[-1].set_xlabel('Ball number (1–40)')
plt.suptitle('Generator frequency vs historical empirical', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig('validation_frequency.png', bbox_inches='tight')
plt.show()

In [ ]:
# NIST SP 800-22 — Frequency (Monobit) and Runs tests
# Bit stream: each number in [1,40] → 6-bit big-endian; powerball [1,10] → 4-bit

def to_bits(tickets):
    bits = []
    for nums, pb in tickets:
        for n in sorted(nums):
            bits.extend([(n >> i) & 1 for i in range(5, -1, -1)])
        bits.extend([(pb >> i) & 1 for i in range(3, -1, -1)])
    return np.array(bits, dtype=int)

def nist_monobit(bits):
    """NIST SP 800-22 section 2.1 — Frequency test."""
    s = np.sum(2 * bits - 1)  # +1/-1 coding
    s_obs = abs(s) / np.sqrt(len(bits))
    p = stats.erfc(s_obs / np.sqrt(2))
    return s_obs, p

def nist_runs(bits):
    """NIST SP 800-22 section 2.3 — Runs test."""
    n = len(bits)
    pi = bits.sum() / n
    if abs(pi - 0.5) >= 2 / np.sqrt(n):
        return None, 0.0  # pre-test fails
    v_obs = 1 + np.sum(bits[:-1] != bits[1:])  # number of runs
    p = stats.erfc(
        abs(v_obs - 2 * n * pi * (1 - pi)) / (2 * np.sqrt(2 * n) * pi * (1 - pi))
    )
    return v_obs, p

print('\n=== NIST SP 800-22 (subset) ===')
print(f'{"Generator":<12} {"Monobit_s":>10} {"Monobit_p":>10} {"Runs":>8} {"Runs_p":>8}')
print('-' * 54)

for label, tickets in [("Decay", decay_tickets), ("Uniform", None), ("Gaussian", gaussian_tickets)]:
    if tickets is None:
        # Reconstruct uniform tickets
        uniform_tickets_tmp = []
        rng2 = np.random.default_rng(seed=99)
        for _ in range(N_SIM):
            nums = sorted(rng2.choice(np.arange(1, 41), size=6, replace=False).tolist())
            pb = int(rng2.integers(1, 11))
            uniform_tickets_tmp.append((nums, pb))
        bits = to_bits(uniform_tickets_tmp)
    else:
        bits = to_bits(tickets)

    s_obs, p_mono = nist_monobit(bits)
    v_obs, p_runs = nist_runs(bits)
    print(f'{label:<12} {s_obs:>10.4f} {p_mono:>10.4f} {str(v_obs):>8} {p_runs:>8.4f}')

print('\n(p > 0.01 indicates pass at α=0.01. Only 2 of 15 NIST tests are run here.)')

## §5 — Discussion & Limits

**What the Poisson/Exponential tests tell us.** If NZ Lotto draws are fair and i.i.d., number appearances are Bernoulli(p=6/40) per draw, which converges to Poisson for large draw counts. Finding that the gap distribution is approximately Exponential and the count-in-window distribution is approximately Poisson is *expected under fairness* — it is not a discovery, it is a confirmation.

**What the decay generator provides.** A frequency-weighted RNG whose sampling distribution directly mirrors historical appearance rates. The decay metaphor is intellectually honest (the math is identical to Poisson emission), and the physical framing helps communicate what the algorithm does. However, because NZ Lotto draws are fair, frequency weights merely reflect sampling noise around uniform — over the long run, no number is truly 'hotter' than another.

**Self-reinforcing bias.** If a number has appeared 3% more than average purely by chance, the decay generator will continue to over-emit it in proportion. This is by design for users who want to mirror recent historical patterns, but it adds no expectation value.

**This analysis does not predict winning numbers.** Lottery outcomes are mechanically random and statistically independent. No analysis of historical draws can confer predictive advantage.

## §6 — References

1. E. Rutherford and H. Geiger (1910). *The Probability Variations in the Distribution of α Particles*. Philosophical Magazine, Series 6, Vol. 20, No. 118, pp. 698–707. https://doi.org/10.1080/14786441008636955

2. J. Walker (1996). *HotBits: Genuine Random Numbers, Generated by Radioactive Decay*. Fourmilab Switzerland. https://www.fourmilab.ch/hotbits/

3. NIST (2018). *SP 800-90B: Recommendation for the Entropy Sources Used for Random Bit Generation*. National Institute of Standards and Technology. https://doi.org/10.6028/NIST.SP.800-90B

4. M. Stipčević and Ç. K. Koç (2014). *True Random Number Generators*. In: Open Problems in Mathematics and Computational Science, Springer, pp. 275–315. https://cetinkayakoc.net/docs/b08.pdf

5. M. Rohe (2003). *RANDy — A True-Random Generator Based On Radioactive Decay*. Semantic Scholar. https://www.semanticscholar.org/paper/RANDy-A-True-Random-Generator-Based-On-Radioactive-Rohe/95d5042fc5963d387563247166b7f904fb1153c8